# Hinode SP: fit both Fe I lines

Download a short interval of calibrated **Level-1 slit scans**, or upload the slit FITS files from one scan. Point the inversion at their folder: the Hinode dataset assembles the raster in memory, using the files' wavelength calibration, pointing, roll and UTC acquisition times. No prepared raster, reference map or intermediate FITS copies are needed.

A Level-2 magnetic-parameter map is a comparison reference, not a Stokes input. The preset shares magnetic/velocity fields across **6301/6302 Å**, fits separate atmosphere parameters and uses a 24.8 mÅ Gaussian spectral response. See the [Hinode guide](https://spinnme.readthedocs.io/en/stable/hinode.html#load-a-downloaded-slit-scan).


## 1. Install and choose a workspace

In Colab, select **Runtime → Change runtime type → GPU** if available. SPINNME detects the device automatically; CPU also works for small trials. Run cells in order. If installation requests a runtime restart, restart and continue here.

Files under `/content` are temporary. Download the results in the last step before ending your session. [Colab runtime limits](https://research.google.com/colaboratory/faq.html#resource-limits) depend on availability.


In [ ]:
# Reuse an existing installation; install only in a fresh Colab runtime.
import importlib.util

if importlib.util.find_spec("spinnme") is None:
    %pip -q install "spinnme"


In [ ]:
import subprocess
import sys
from pathlib import Path

import torch


def spinnme(*arguments):
    subprocess.run([sys.executable, "-m", "spinnme", *map(str, arguments)], check=True)

ROOT = Path("/content/spinnme/hinode")
ROOT.mkdir(parents=True, exist_ok=True)
RAW = ROOT / "raw"
print("Workspace:", ROOT)
print("CUDA available:", torch.cuda.is_available())
spinnme("--version")


## 2. Download or upload calibrated slit scans

The example downloads two minutes of a Hinode scan to keep this trial small. Edit the UTC interval for your observation. Set `DOWNLOAD_FROM_ARCHIVE=False` to upload multiple calibrated slit FITS files from one scan instead. Files go directly into one input folder; there is no crop or preparation step.


In [ ]:
DOWNLOAD_FROM_ARCHIVE = True  # @param {type:"boolean"}
START = "2024-03-23T02:00:35"  # @param {type:"string"}
END = "2024-03-23T02:02:35"  # @param {type:"string"}

if DOWNLOAD_FROM_ARCHIVE:
    spinnme("download", "hinode", "--start", START, "--end", END, "--output", RAW)
else:
    from google.colab import files
    RAW.mkdir(parents=True, exist_ok=True)
    uploaded = files.upload()
    if not uploaded or any(not name.lower().endswith((".fits", ".fits.gz")) for name in uploaded):
        raise ValueError("Upload calibrated slit FITS files from one scan.")
    for name, contents in uploaded.items():
        (RAW / Path(name).name).write_bytes(contents)
    del uploaded, contents


In [ ]:
from astropy.io import fits

slits = sorted(RAW.glob("*.fits")) + sorted(RAW.glob("*.fits.gz"))
if not slits:
    raise ValueError("No slit FITS files found in the input folder.")
first_header = fits.getheader(slits[0])
print("Slit folder:", RAW)
print("Slit files:", len(slits))
print("First acquisition:", first_header.get("DATE-OBS", first_header.get("DATE_OBS")))


## 3. Create a configuration dictionary and fit

This configuration matches the full `hinode/sp` preset and evaluation inversion: a 512-wide, 8-layer network with isotropic positional encoding, batches of 8192, SOAP, ME solver compilation and 200,000 steps. Validation runs every 2000 batches. It uses a 36,000-second time scale, source-function ratios and linear Stokes scaling. These settings do not guarantee convergence; inspect the spectra below. W&B is disabled; no account is needed.

For a new setup use a new `RUN_NAME`. To continue a compatible checkpoint, set `RESUME=True` and increase `MAX_STEPS`. Reuse the downloaded slit folder when resuming. Changing the input files requires a fresh run/cache; there is no assembled FITS file to manage.

Both lines use ±0.30 Å fitting windows and a Gaussian instrumental response with 24.8 mÅ FWHM. Nine quadrature nodes integrate each synthetic spectral sample while retaining the measured wavelength grid and native slit positions.

Validation checks both lines and places them in separate wavelength panels in **Stokes profiles - Comparison**.


In [ ]:
RUN_NAME = "trial-01"  # Change this for a different scientific setup.
MAX_STEPS = 200000
RESUME = False  # Set True only to resume the same run/checkpoint.
RUN = ROOT / "runs" / RUN_NAME
PRODUCTS = RUN / "products"


In [ ]:
# All inversion settings are editable here; pass this dictionary directly to run().
config = {'path': str(RUN),
 'work_path': str(RUN / "cache"),
 'data': {'train_configs': [{'type': 'hinode',
                             'data_path': str(RAW),
                             'wavelength_center': 6301.5012,
                             'stokes_normalization': 1.0,
                             'instrument_id': 'HINODE_6301',
                             'line_id': 'fe-i-6301',
                             'time_reference': 'observation',
                             'wavelength_range': [6301.2012, 6301.8012]},
                            {'type': 'hinode',
                             'data_path': str(RAW),
                             'wavelength_center': 6302.4931,
                             'stokes_normalization': 1.0,
                             'instrument_id': 'HINODE_6302',
                             'line_id': 'fe-i-6302',
                             'time_reference': 'observation',
                             'wavelength_range': [6302.1931, 6302.7931]}],
          'num_workers': 4,
          'num_dataset_workers': 16,
          'batch_size': 8192,
          'dataset_batch_size': 512,
          'seconds_per_dt': 36000,
          'gauss_per_dB': 1000,
          'valid_config': [{'type': 'hinode',
                           'data_path': str(RAW),
                           'wavelength_center': 6302.4931,
                           'stokes_normalization': 1.0,
                           'instrument_id': 'HINODE_6302',
                           'line_id': 'fe-i-6302',
                           'time_reference': 'observation',
                           'wavelength_range': [6302.1931, 6302.7931],
                           'sample_idx': 0},
                          {'type': 'hinode',
                             'data_path': str(RAW),
                             'wavelength_center': 6301.5012,
                             'stokes_normalization': 1.0,
                             'instrument_id': 'HINODE_6301',
                             'line_id': 'fe-i-6301',
                             'time_reference': 'observation',
                             'wavelength_range': [6301.2012, 6301.8012], 'sample_idx': 0}],
          'ref_time': None},
 'instrument': [{'type': 'hinode',
                 'instrument_id': 'HINODE_6301',
                 'limb_correction': False,
                 'velocity_correction': False,
                 'artifact_correction': False,
                 'line_id': 'fe-i-6301',
                 'spectral_response': {'type': 'gaussian', 'fwhm': 0.0248,
                                       'unit': 'Angstrom', 'samples': 9}},
                {'type': 'hinode',
                 'instrument_id': 'HINODE_6302',
                 'limb_correction': False,
                 'velocity_correction': False,
                 'artifact_correction': False,
                 'line_id': 'fe-i-6302',
                 'spectral_response': {'type': 'gaussian', 'fwhm': 0.0248,
                                       'unit': 'Angstrom', 'samples': 9}}],
 'model': {'type': 'mlp',
           'dim': 512,
           'n_layers': 8,
           'vector_potential': False,
           'source_function': 'ratio',
           'activation': 'swish',
           'encoding_config': {'type': 'positional_isotropic',
                               'num_frequencies': [256, 32],
                               'min_frequencies': [0, 0],
                               'max_frequencies': [12, 4],
                               'include_input': False}},
 'training': {'plot_validation': True,
              'epochs': -1,
              'max_steps': MAX_STEPS,
              'check_val_every_n_epoch': None,
              'val_check_interval': max(1, min(MAX_STEPS, 2000)),
              'lr_params': {'start': 0.001, 'end': 0.0001, 'iterations': MAX_STEPS},
              'optimizer_config': {'type': 'soap'},
              'compile_config': {'me_solver': True, 'mode': 'default'}},
 'normalization': {'asinh_alphas': None,
                   'stokes_scales': {'I': 1.0, 'Q': 0.1, 'U': 0.1, 'V': 0.1}},
 'stokes_loss': {'type': 'mse', 'continuum_normalization': True, 'continuum_quantile': 0.9},
 'weights': {'I': 1.0, 'Q': 1.0, 'U': 1.0, 'V': 1.0},
 'logging': {'project': 'spinnme', 'name': RUN_NAME, 'mode': 'disabled'}}


In [ ]:
from spinnme import run

module, data_module, trainer = run(config, resume=RESUME)
SNAPSHOT = RUN / "inversion.pme"


## 4. Export the fitted maps

Load the `.pme` once, then write **one FITS per parameter**, an NPZ bundle and JPG previews. `Brtp` expands to `Br`, `Btheta`, `Bphi`; `Blos` is positive toward the observer and `Vdop` is positive redshift in m/s. FITS/NPZ retain masks, timing and native slit coordinates in provenance; their primary WCS is a nominal display grid. JPGs are quick looks. No azimuth disambiguation is applied.

The JSON/CSV summaries describe the fitted maps, not accuracy against truth. The spectral check in the next step assesses the fit.


In [ ]:
from spinnme import load
from spinnme.evaluation import evaluate_result, write_metrics

maps, spectra = {}, {}
with load(SNAPSHOT) as snapshot:
    saved = [row for row in snapshot.observations if row["role"] == "train"]
    loss_settings = dict(snapshot.stokes_loss_config)
    for row in saved:
        identifier = row["id"]
        parameters = ["Brtp", "B", "Blos", "Vdop", "vmac", "damping"]
        if config["instrument"][0].get("artifact_correction"):
            parameters.append("Artifact")
        result = snapshot.evaluate(identifier, parameters=parameters)
        own = PRODUCTS / identifier
        result.write_fits(own / "fits", overwrite=True)
        result.write_npz(own / "parameters.npz", overwrite=True)
        result.write_jpg(own / "preview", overwrite=True)
        write_metrics(evaluate_result(result), own / "summary.json")
        maps[identifier] = result
        spectra[identifier] = snapshot.evaluate(identifier, parameters=["IQUV"])
        print(identifier, row["instrument_id"], row["shape"], row["time"]["model_time"])
print("Exports:", PRODUCTS)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from astropy.wcs import WCS

first_id = saved[0]["id"]
result = maps[first_id]
figure, axes = plt.subplots(1, 2, figsize=(10, 4),
                           subplot_kw={"projection": WCS(result.header).celestial})
for axis, key in zip(axes, ["Blos", "Vdop"]):
    values = result.arrays[key]
    limit = max(float(np.nanpercentile(np.abs(values), 99)), 1.0)
    image = axis.imshow(values, origin="lower", cmap="RdBu_r", vmin=-limit, vmax=limit)
    axis.set_title(key)
    figure.colorbar(image, ax=axis, label=result.units[key])
figure.tight_layout()
figure.savefig(PRODUCTS / "field-preview.png", dpi=150)
plt.show()


## 5. Check observed versus fitted Stokes

Inspect all four components and look for structured residuals. The table reports RMS residuals over valid pixels/wavelengths in the plotted intensity normalization, **before loss scaling**. It is not reduced chi-squared or a parameter uncertainty. With continuum-normalized fits, each observed/synthetic profile uses its own intensity quantile. Compare residuals with noise estimated in the same normalization.

This is a fit check on training data. Use held-out acquisitions to assess interpolation; one acquisition does not establish temporal evolution. Both Hinode lines are checked below.


In [ ]:
from spinnme.data.hinode import read_hinode_slits


def read_observed(row):
    dataset = next(entry for entry in config["data"]["train_configs"]
                   if entry["instrument_id"] == row["instrument_id"])
    return read_hinode_slits(row["source_paths"], line_id=row["line_id"],
        wavelength_center=dataset["wavelength_center"],
        wavelength_range=dataset["wavelength_range"])["stokes"]


In [ ]:
import json

rms_rows = []
for row in saved:
    identifier = row["id"]
    fitted = spectra[identifier]
    measured = read_observed(row) / row["stokes_normalization"]
    if loss_settings.get("continuum_normalization", False):
        quantile = loss_settings.get("continuum_quantile", 0.9)
        continuum = np.quantile(measured[..., 0:1, :], quantile, axis=-1, keepdims=True)
        measured = measured / np.where(continuum > 0, continuum, np.nan)
    predicted = np.stack([fitted.arrays[key] for key in "IQUV"], axis=-2)
    if measured.shape != predicted.shape:
        raise ValueError("Observed and fitted grids/wavelength selections must match.")
    valid = fitted.mask & np.isfinite(measured).all(axis=(-2, -1))
    if not valid.any() or not np.isfinite(predicted[valid]).all():
        raise ValueError("No valid comparison or non-finite fitted spectra.")
    residual = predicted - measured
    rms = np.sqrt(np.mean(residual[valid] ** 2, axis=(0, 2)))
    rms_rows.append({"observation": identifier, "instrument": row["instrument_id"],
                     **dict(zip("IQUV", map(float, rms)))})
    print(identifier, "I/Q/U/V RMS:", rms)
    y, x = np.argwhere(valid)[len(np.argwhere(valid)) // 2]
    wavelength = (np.asarray(fitted.metadata["wavelength_grid_m"])
                  + fitted.metadata["wavelength_center_m"]) * 1e10
    figure, axes = plt.subplots(2, 4, figsize=(12, 5), sharex=True)
    for index, component in enumerate("IQUV"):
        axes[0, index].plot(wavelength, measured[y, x, index], "o", label="observed")
        axes[0, index].plot(wavelength, predicted[y, x, index], label="fitted")
        axes[0, index].set_title(component)
        axes[1, index].plot(wavelength, residual[y, x, index])
        axes[1, index].axhline(0, color="k", linewidth=0.5)
        axes[1, index].set_xlabel("Wavelength [Å]")
    axes[0, 0].set_ylabel("Stokes intensity")
    axes[1, 0].set_ylabel("Fitted − observed")
    axes[0, 0].legend()
    figure.suptitle(identifier)
    figure.tight_layout()
    figure.savefig(PRODUCTS / identifier / "stokes-fit.png", dpi=150)
    plt.show()
(PRODUCTS / "stokes-rms.json").write_text(json.dumps(rms_rows, indent=2))


## 6. Save your results

Download the snapshot, resolved configuration/checkpoint, metrics and exported products. Input observations, calibration files and the large preparation cache are excluded. A `.pme` can export its saved grids without the original observations; resuming training also needs the same inputs/calibration. Keep those separately if you plan to resume after the Colab runtime ends.


In [ ]:
import shutil

bundle = ROOT / "download" / RUN_NAME
bundle.mkdir(parents=True, exist_ok=True)
for name in ["inversion.pme", "last.ckpt", "resolved.yaml", "run_manifest.json"]:
    shutil.copy2(RUN / name, bundle / name)
shutil.copytree(PRODUCTS, bundle / "products", dirs_exist_ok=True)
archive = shutil.make_archive(str(ROOT / RUN_NAME), "zip", bundle)
try:
    from google.colab import files
except ImportError:
    print("Results ZIP:", archive)
else:
    files.download(archive)


Next: [training options](https://spinnme.readthedocs.io/en/stable/training_options.html), [fit quality](https://spinnme.readthedocs.io/en/stable/evaluation.html), [Python exports at custom resolution/cadence](https://spinnme.readthedocs.io/en/stable/python_exports.html), or the [beginner FAQ](https://spinnme.readthedocs.io/en/stable/faq.html).
